# Tutorial 4 — Build a Star Schema and Explore the Cube with OLAP
**Z3003 Data Mining: Scaling to Large Data** · IIT Madras Zanzibar · Thursday 8 October 2026, 4:45–5:30 pm · Dr. Innocent Nyalala

**Dataset this week:** `datasets/zanzibar_farm_seasons.csv` — **200,000 rows** = 10,000 farms × 10 years (2016–2025) × 2 seasons (Masika = long rains, Vuli = short rains). Synthetic teaching data built from the same patterns as our 500-farm sample (see `datasets/DATA_DICTIONARY.md`).

| Part | Time | Task | Lecture |
|---|---|---|---|
| A | 5 min | Load, inspect and clean 200,000 rows | Week 2 |
| B | 10 min | Build a **star schema**: one fact table + dimension tables | Lecture 10 |
| C | 15 min | The five **OLAP operations**: roll-up, drill-down, slice, dice, pivot | Lecture 11 |
| D | 7 min | All 8 **cuboids**, and the averaging trap | Lectures 10 & 12 |
| E | 8 min | Graded exercise | — |

**How to work:** read each explanation, then complete the code under it. Every task ends with a ✅ **Checkpoint** — if your number differs, fix it before moving on. `# TODO` = your code. ✍️ = a written answer in your own words.

## Part A — Load and clean
Two known problems (from `DATA_DICTIONARY.md`): some yields are missing, and **25 rain-gauge readings are the error code `9999`**. An error code is not a real value — replace it with `NaN` (missing) so it cannot distort averages.

In [ ]:
import pandas as pd
import numpy as np
import itertools, time

df = pd.read_csv("datasets/zanzibar_farm_seasons.csv")
print(df.shape)
df.head(3)

In [ ]:
print("rain-gauge errors (9999):", (df["rainfall_mm"] == 9999).sum())
# TODO: replace every 9999 in rainfall_mm with np.nan
print("missing values per column after cleaning:")
print(df.isna().sum()[df.isna().sum() > 0])

✅ **Checkpoint A:** shape `(200000, 17)`; 25 errors; after cleaning, `rainfall_mm` has **2,021** missing (1,996 + 25) and `yield_tonnes` **8,111**.
> pandas' `sum()` and `mean()` skip `NaN` automatically, so totals below use only recorded values.

## Part B — Build a star schema (Lecture 10)
A **star schema** = one **fact table** (one row per event, holding measures + a key for each dimension) and one small **dimension table** per dimension.
We build: `dim_location` (district → region), `dim_time` (year, season), `dim_crop` (crop → crop group).

### B1. Dimension tables

In [ ]:
dim_location = df[["district", "region"]].drop_duplicates().sort_values(["region", "district"]).reset_index(drop=True)
dim_location["location_key"] = np.arange(1, len(dim_location) + 1)

# TODO: build dim_time the same way from ["year", "season"], with a time_key column
dim_time = None

groups = {"Clove": "cash crop", "Cashew": "cash crop", "Rice": "food crop", "Cassava": "food crop", "Coconut": "food crop"}
# TODO: build dim_crop from the unique crops, add crop_group using the dictionary above, and a crop_key
dim_crop = None

print(len(dim_location), len(dim_time), len(dim_crop))
dim_crop

✅ **Checkpoint B1:** **11** locations, **20** time periods, **5** crops.

### B2. The fact table: replace text with keys
`merge` looks up each row's key in a dimension table. Then keep only keys + measures.

In [ ]:
fact = (df.merge(dim_location, on=["district", "region"])
          # TODO: merge dim_time on ["year", "season"] and dim_crop on "crop"
        )
fact = fact[["record_id", "farm_id", "location_key", "time_key", "crop_key",
             "yield_tonnes", "revenue_usd", "rainfall_mm", "price_per_kg"]]
print(fact.shape)
print("memory, full table (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print("memory, fact table (MB):", round(fact.memory_usage(deep=True).sum() / 1e6, 1))

✅ **Checkpoint B2:** fact table `(200000, 9)`, and far less memory than the full table — **keys instead of repeated text**.

### B3. A star-schema query: total clove revenue by region in 2023
Join the fact table back to the dimensions you need, filter, group, sum — one join per dimension (Lecture 10).

In [ ]:
q = (fact.merge(dim_location, on="location_key")
         .merge(dim_crop, on="crop_key")
         .merge(dim_time, on="time_key"))
# TODO: keep crop == "Clove" and year == 2023, then total revenue_usd by region, in $ millions


✅ **Checkpoint B3:** Pemba South is the largest (≈ **23.2** $M — the cube cell shown on Wednesday's slides).

## Part C — The five OLAP operations (Lecture 11)
From here we use the cleaned `df` directly (it already contains every dimension column).

### C1. Roll-up: district → region

In [ ]:
by_district = df.groupby(["region", "district"])["revenue_usd"].sum() / 1e6
# TODO: roll up to region level (sum over districts) and print both Pemba South lines


✅ **Checkpoint C1:** Chake Chake **123.0** + Mkoani **122.5** = Pemba South **245.6** $M. SUM is *distributive*.

### C2. Drill-down: year → season (find the drought)

In [ ]:
# TODO: average yield by year; then drill down to (year, season) for 2016–2018


✅ **Checkpoint C2:** 2017 averages **2.22 t** (vs ≈ 2.7 normally); drilled down, **both** seasons fell — Masika 2.39, Vuli 2.05. An island-wide drought.

### C3. Slice: year = 2017

In [ ]:
# TODO: slice the cube at year == 2017 and show average yield as a region × crop table


✅ **Checkpoint C3:** every region is low for every crop (e.g. clove ≈ 1.56 t vs 1.85 t normally).

### C4. Dice: Pemba × Clove × 2021–2025

In [ ]:
# TODO: keep region in Pemba North / Pemba South, crop == Clove, year between 2021 and 2025
dice = None
print("rows with a recorded revenue:", dice["revenue_usd"].count())
print("total revenue ($M):", round(dice["revenue_usd"].sum() / 1e6, 1))

✅ **Checkpoint C4:** **11,862** rows with revenue; **175.8** $M.

### C5. Pivot, with ALL totals (like SQL `ROLLUP`)

In [ ]:
# TODO: pivot_table of revenue ($M) with crop as rows and year as columns, margins=True, margins_name="ALL"


✅ **Checkpoint C5:** the bottom-right `ALL, ALL` cell is the grand total ≈ **986.0** $M. Now pivot it: swap `index` and `columns` — the numbers do not change, only the reading.

## Part D — Cuboids and the averaging trap
### D1. Compute all 2³ = 8 cuboids of (region, crop, year)

In [ ]:
dims = ["region", "crop", "year"]
cuboids = {}
for k in range(3, -1, -1):
    for combo in itertools.combinations(dims, k):
        # TODO: if combo is empty, store the grand total; else groupby(list(combo)) and sum revenue_usd
        pass
for combo, table in cuboids.items():
    print(f"{str(combo):28s} {len(table) if hasattr(table, '__len__') else 1:4d} cells")

✅ **Checkpoint D1:** 250, 25, 50, 50, 5, 5, 10, 1 cells — the lattice from Wednesday's slides.

### D2. Pre-computing pays: answer 'revenue by region' two ways

In [ ]:
t0 = time.perf_counter(); a = df.groupby("region")["revenue_usd"].sum(); t_base = time.perf_counter() - t0
rc = cuboids[("region", "crop")]
t0 = time.perf_counter(); b = rc.groupby(level="region").sum(); t_pre = time.perf_counter() - t0
print("same answer:", np.allclose(a.sort_index(), b.sort_index()))
print(f"from 200,000 rows: {t_base*1000:.2f} ms   from the 25-cell cuboid: {t_pre*1000:.2f} ms")

✅ **Checkpoint D2:** `same answer: True`, and the stored cuboid is faster (exact times vary by laptop). This is **partial materialization** (Friday).

### D3. The averaging trap (Lecture 10)

In [ ]:
crop_avg = df.groupby("crop")["yield_tonnes"].mean()
print(crop_avg.round(3))
# TODO: print (a) the plain average of the five crop averages, and (b) the true average yield over all rows


✅ **Checkpoint D3:** average of averages **3.088** t vs true **2.700** t. ✍️ **Why so different?** (Hint: count the farms per crop with `value_counts()`.)

*Your answer:*

## Part E — Graded exercise (8 min)
**E1. Roll-up along the crop hierarchy.** Total revenue ($M) by `crop_group` (cash crop vs food crop). Use the `groups` dictionary from B1.

In [ ]:
# E1: your code


**E2. Iceberg cube.** Group by `district, crop, year, season` and count records. How many of the cells have **at least 100** records?

In [ ]:
# E2: your code


**E3. Bitmap index.** Build two True/False arrays (bitmaps): `crop == "Clove"` and `region == "Pemba South"`. Combine them with `&` and count the rows.

In [ ]:
# E3: your code


✍️ **E4.** In 2–3 sentences: a manager sees that total revenue in **2017** is unusually low. Which OLAP operations would you use, in which order, to explain it? Quote at least one number from Part C.

*Your answer:*

---
### Submission
Run all cells so outputs are visible, save as **`YourName_W4_Tutorial4.ipynb`**, and upload before **5:30 pm**.
### Mini-project
Dataset approval checkpoint: **Friday 16 October**. If your dataset has a time or location column, today's star schema and OLAP code is a strong start for your exploratory analysis.